# Lab 3 Track B: NorthStar Policy-Grounded Retention Offer

This notebook demonstrates the Track B path from customer profile and policy corpus to retrieved policy passages and a generated retention message. It uses the local FAISS index and one Bedrock offer-generation call. RAGAS scoring is kept in the separate evaluation harness so this demo does not repeat the more expensive judge calls.

## Design choices

- **Embeddings:** Amazon Titan Text Embeddings V2, 256 dimensions, normalized. It integrates with the existing Bedrock access and is inexpensive for this small corpus.
- **Vector store:** FAISS `IndexFlatIP` locally. The corpus has only about 60 chunks, so a managed vector service would add cost and operations without a clear need.
- **Chunking:** 600-character chunks with 100-character overlap. Splitting stays within Markdown sections, and overlap preserves nearby policy details at boundaries. These are character counts.
- **Generation:** Claude Haiku 4.5 through the Bedrock inference profile, suitable for a short offer at low per-call cost.
- **Reranking:** Not used. For this small corpus, two focused retrieval passes (tier benefits and policy constraints), followed by de-duplication, keep retrieval simple. The separate strategy comparison measures the tradeoff.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'rag').is_dir() and (p / 'northstar-policy-docs').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Could not locate the cs401r-m0212 repository root from the notebook working directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rag.build_index import (
    CHUNK_OVERLAP, CHUNK_SIZE, EMBEDDING_DIMENSIONS, EMBEDDING_MODEL_ID,
    build_local_faiss_index, load_local_faiss_index, load_policy_chunks,
)

print(f'Repository: {ROOT}')
print(f'Embedding model: {EMBEDDING_MODEL_ID} ({EMBEDDING_DIMENSIONS} dimensions)')
print(f'Chunking: {CHUNK_SIZE} characters, {CHUNK_OVERLAP} character overlap')

Repository: C:\Users\maema\cs401R\cs401r-m0212
Embedding model: amazon.titan-embed-text-v2:0 (256 dimensions)
Chunking: 600 characters, 100 character overlap


In [2]:
# Reuse the local index when present. If a fresh checkout has no index, build it from the policy Markdown files.
# The local_index directory is git-ignored; that first build makes one Titan embedding request per chunk.
index_dir = ROOT / 'rag' / 'local_index'
index_file = index_dir / 'policies.faiss'
chunks_file = index_dir / 'chunks.json'

if not (index_file.exists() and chunks_file.exists()):
    policy_chunks = load_policy_chunks(
        ROOT / 'northstar-policy-docs',
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP,
    )
    build_local_faiss_index(policy_chunks, index_dir=index_dir)
    print(f'Built FAISS index from {len(policy_chunks)} policy chunks.')
else:
    print('Reusing the saved local FAISS index.')

index, chunks = load_local_faiss_index(index_dir)
print(f'Loaded {index.ntotal} vectors and {len(chunks)} matching policy chunks.')

Reusing the saved local FAISS index.
Loaded 60 vectors and 60 matching policy chunks.


## Run the RAG offer flow

The retriever makes one tier-benefit pass and one constraints pass, then removes duplicate chunks. The generator receives the customer profile for personalization and retrieved policy text for policy claims. The profile is not treated as evidence for policy benefits or tier changes.

In [3]:
from rag.generate_offer import MODEL_ID, REGION, rag_invoke

customer = {
    'churn_probability': 0.81,
    'loyalty_tier': 'Gold',
    'days_since_last_purchase': 47,
    'total_lifetime_value': 3120.40,
    'avg_order_value': 186.25,
    'category_diversity_score': 0.5,
    'online_to_store_ratio': 0.72,
    'top_categories': ['Outdoor', 'Footwear'],
}
question = (
    'Generate a personalized retention offer for this at-risk Gold member '
    'who shops mostly online for outdoor gear and footwear.'
)

result = rag_invoke(customer, question)
policy_contexts = [c for c in result['contexts'] if c.startswith('Source: ')]

print(f'Generation model: {MODEL_ID} ({REGION})')
print(f'Retrieved policy passages: {len(policy_contexts)}')
for n, context in enumerate(policy_contexts, start=1):
    print(f'{n}. {context.splitlines()[0]}')

print('\nQuestion:')
print(question)
print('\nGenerated offer:')
print(result['answer'])

Generation model: us.anthropic.claude-haiku-4-5-20251001-v1:0 (us-east-1)
Retrieved policy passages: 6
1. Source: loyalty-program-terms.md | Section: NorthStar Summit Rewards — Program Terms > 4. Tier Benefits
2. Source: customer-faq.md | Section: NorthStar Retail — Customer Service FAQ > Loyalty
3. Source: customer-faq.md | Section: NorthStar Retail — Customer Service FAQ > Loyalty
4. Source: loyalty-program-terms.md | Section: NorthStar Summit Rewards — Program Terms > 5. Promotional Offers > 5.1 Retention Offers
5. Source: loyalty-program-terms.md | Section: NorthStar Summit Rewards — Program Terms > 5. Promotional Offers
6. Source: return-policy.md | Section: NorthStar Retail — Returns and Exchanges Policy > 4. Non-Returnable Merchandise

Question:
Generate a personalized retention offer for this at-risk Gold member who shops mostly online for outdoor gear and footwear.

Generated offer:
**Subject:** Your Gold Benefits Are Waiting—Free Shipping & Returns on Your Next Order

**Body:

## Review

The saved output above records one end-to-end run. Check that policy benefits in the message are supported by the retrieved passages, customer categories come from the supplied profile, and the answer does not invent discounts, deadlines, scarcity, or tier-maintenance claims. The separate RAGAS harness evaluates faithfulness, answer relevance, and context recall across all four provided test cases.